In [2]:
import os
import shutil
from pathlib import Path

# Possible original dirs
orig_img_dirs = [
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\images\test",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\images\train",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\images\val",
]
orig_label_dirs = [
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\labels\test",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\labels\train",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\labels\val",
]

adv_test = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\patched_random_bigger_test\images"

dest_img = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\clean_test\images"
dest_labels = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\clean_test\labels"

# Ensure destination directories exist
os.makedirs(dest_img, exist_ok=True)
os.makedirs(dest_labels, exist_ok=True)

# Collect adv_test image names
adv_images = [Path(f).stem for f in os.listdir(adv_test) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

copied = 0
missing = []

for name in adv_images:
    found = False
    for img_dir, lbl_dir in zip(orig_img_dirs, orig_label_dirs):
        src_img_jpg = Path(img_dir) / f"{name}.jpg"
        src_img_png = Path(img_dir) / f"{name}.png"
        src_label = Path(lbl_dir) / f"{name}.txt"

        if src_img_jpg.exists() or src_img_png.exists():
            src_img = src_img_jpg if src_img_jpg.exists() else src_img_png

            # Copy image
            shutil.copy2(src_img, Path(dest_img) / src_img.name)

            # Copy label if exists
            if src_label.exists():
                shutil.copy2(src_label, Path(dest_labels) / src_label.name)
            else:
                print(f"⚠️ No label for {name}")

            copied += 1
            found = True
            break  # stop searching once found in one split

    if not found:
        missing.append(name)

print(f"✅ Done! Copied {copied} images.")
if missing:
    print("❌ Missing originals for:", missing)


✅ Done! Copied 204 images.


## Load necessary library

In [3]:
import os
from ultralytics import YOLO
from pathlib import Path

In [4]:
model = YOLO("yolov5s.pt")
# tjudhd_model = YOLO("./tju-dhd.pt")

PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



## Original vs Adversarial mAP50 (COCO-Pretrained)

In [50]:
img_dir_clean = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched"

results = model.val(data=os.path.join(img_dir_clean, "data_win_tsea2_inpainted_xgb.yaml"), split="test", imgsz=512)

Ultralytics 8.3.189  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
val: Fast image access  (ping: 0.40.0 ms, read: 27.35.7 MB/s, size: 312.9 KB)
val: Scanning C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\XGB_Inpainted\TSEA2\labels... 80 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 309.0it/s 0.3s
val: New cache created: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\XGB_Inpainted\TSEA2\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.00it/s 5.0s
                   all         80        446      0.597      0.493      0.548      0.241
                person         80        446      0.597      0.493      0.548      0.241
Speed: 1.4ms preprocess, 8.4ms inference, 0.1ms loss, 9.3ms postprocess per image
Results saved to C:\Adrianov\Projects\Project-Satanael\runs\detect\val77


## Original vs Adversarial mAP50 (COCO-Pretrained)

In [39]:
!pip install -r https://raw.githubusercontent.com/ultralytics/yolov5/master/requirements.txt


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
sys.path.insert(0, r'C:\Adrianov\Projects\yolov5')  # path to your local YOLOv5 repo

from val import run

# Validate your YOLOv5 model on class 0
results = run(
    weights='tju-dhd.pt',
    data=r'C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\data_win_tsea2.yaml',
    imgsz=512,
    task='test'
)

print(results)

C:\Adrianov\Projects\yolov5\utils\general.py:33: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg
YOLOv5  v7.0-418-ga493afe1 Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)

C:\Adrianov\Projects\yolov5\models\experimental.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary ob

FileNotFoundError: [Errno 2] No such file or directory: 'tju-dhd.pt'

## End-to-End PAD

In [2]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2
import sys
import glob
import os
import importlib.util
from segment_anything import SamPredictor, SamAutomaticMaskGenerator, sam_model_registry

import math
from PIL import Image

In [3]:
fusefilter_path = os.path.abspath("../../defenselib/fuse_filter.py")

spec = importlib.util.spec_from_file_location("fuse_filter", fusefilter_path)
fusefilter = importlib.util.module_from_spec(spec)
sys.modules["fusefilter"] = fusefilter
spec.loader.exec_module(fusefilter)


In [4]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="The number of unique classes is greater than 50%",
    category=UserWarning
)


In [5]:
def save_heatmap(img, path, cmap='grey'):
    plt.imsave(path, img, cmap=cmap)

In [6]:
processed_files = ['Naturalistic1_1499917703635.jpg',
 'Naturalistic1_1499918339461.jpg',
 'Naturalistic1_1499923500376.jpg',
 'Naturalistic1_1499925371739.jpg',
 'Naturalistic1_1499931818829.jpg',
 'Naturalistic1_1499932367119.jpg',
 'Naturalistic1_1499933971246.jpg',
 'Naturalistic1_1499934217508.jpg',
 'Naturalistic1_1499934495500.jpg',
 'Naturalistic1_1499939610334.jpg',
 'Naturalistic1_1499940414097.jpg',
 'Naturalistic1_1499940503860.jpg',
 'Naturalistic1_1499940647224.jpg',
 'Naturalistic1_1499991199874.jpg',
 'Naturalistic1_1499995387796.jpg',
 'Naturalistic1_1499997243748.jpg',
 'Naturalistic1_1499997506250.jpg',
 'Naturalistic1_1499997891867.jpg',
 'Naturalistic1_1499998395606.jpg',
 'Naturalistic1_1499998680556.jpg',
 'Naturalistic1_1499998720024.jpg',
 'Naturalistic1_1499999051151.jpg',
 'Naturalistic1_1499999078373.jpg',
 'Naturalistic1_1500000581873.jpg',
 'Naturalistic1_1500001935862.jpg',
 'Naturalistic1_1500002615319.jpg',
 'Naturalistic1_1500004891513.jpg',
 'Naturalistic1_1500004915796.jpg',
 'Naturalistic1_1500005214593.jpg',
 'Naturalistic1_1500005237964.jpg',
 'Naturalistic1_1500005395445.jpg',
 'Naturalistic1_1500006360560.jpg',
 'Naturalistic1_1500006810696.jpg',
 'Naturalistic1_1500016129976.jpg',
 'Naturalistic1_1502180481674.jpg',
 'Naturalistic1_1502183697354.jpg',
 'Naturalistic1_1502190352466.jpg',
 'Naturalistic1_1502190398627.jpg',
 'Naturalistic1_1502239969934.jpg',
 'Naturalistic1_1502241775513.jpg',
 'Naturalistic1_1502242609181.jpg',
 'Naturalistic1_1502256560672.jpg',
 'Naturalistic1_1502267602199.jpg',
 'Naturalistic1_1502267613821.jpg',
 'Naturalistic1_1502269919428.jpg',
 'Naturalistic1_1502353296551.jpg',
 'Naturalistic1_1502362055473.jpg',
 'Naturalistic1_1502363129220.jpg',
 'Naturalistic1_1502430313322.jpg',
 'Naturalistic1_1502437185415.jpg',
 'Naturalistic2_1499933007258.jpg',
 'Naturalistic2_1499934079635.jpg',
 'Naturalistic2_1499935363579.jpg',
 'Naturalistic2_1499937456376.jpg',
 'Naturalistic2_1499940363834.jpg',
 'Naturalistic2_1499940660827.jpg',
 'Naturalistic2_1499940750044.jpg',
 'Naturalistic2_1499994273704.jpg',
 'Naturalistic2_1499997533191.jpg',
 'Naturalistic2_1499998733628.jpg',
 'Naturalistic2_1499998901468.jpg',
 'Naturalistic2_1499999188181.jpg',
 'Naturalistic2_1499999242688.jpg',
 'Naturalistic2_1499999536764.jpg',
 'Naturalistic2_1500001377584.jpg',
 'Naturalistic2_1500001781874.jpg',
 'Naturalistic2_1500001835164.jpg',
 'Naturalistic2_1500001898453.jpg',
 'Naturalistic2_1500002666285.jpg',
 'Naturalistic2_1500004867400.jpg',
 'Naturalistic2_1500004879323.jpg',
 'Naturalistic2_1500005321965.jpg',
 'Naturalistic2_1500005515768.jpg',
 'Naturalistic2_1500014120470.jpg',
 'Naturalistic2_1500015073522.jpg',
 'Naturalistic2_1502179636573.jpg',
 'Naturalistic2_1502183525879.jpg',
 'Naturalistic2_1502184493065.jpg',
 'Naturalistic2_1502188451026.jpg',
 'Naturalistic2_1502188497186.jpg',
 'Naturalistic2_1502190363979.jpg',
 'Naturalistic2_1502190387114.jpg',
 'Naturalistic2_1502235046404.jpg',
 'Naturalistic2_1502239946909.jpg',
 'Naturalistic2_1502247156329.jpg',
 'Naturalistic2_1502247327727.jpg',
 'Naturalistic2_1502251037884.jpg',
 'Naturalistic2_1502259901730.jpg',
 'Naturalistic2_1502261579575.jpg',
 'Naturalistic2_1502268805445.jpg',
 'Naturalistic2_1502272166221.jpg',
 'Naturalistic2_1502333554745.jpg',
 'Naturalistic2_1502361874129.jpg',
 'Naturalistic2_1502362718082.jpg',
 'Naturalistic2_1502363108718.jpg',
 'Naturalistic2_1502411443356.jpg',
 'Naturalistic2_1502411558375.jpg',
 'Naturalistic2_1502431265380.jpg',
 'Naturalistic2_1502438014353.jpg',
 'Naturalistic2_1502444817510.jpg',
 'Naturalistic3_1499997303996.jpg',
 'Naturalistic3_1499997442961.jpg',
 'Naturalistic3_1499997995091.jpg',
 'Naturalistic3_1499998047897.jpg',
 'Naturalistic3_1499998329571.jpg',
 'Naturalistic3_1499998706702.jpg',
 'Naturalistic3_1499998786621.jpg',
 'Naturalistic3_1499999435925.jpg',
 'Naturalistic3_1500002543238.jpg',
 'Naturalistic3_1500002760259.jpg',
 'Naturalistic3_1500002954058.jpg',
 'Naturalistic3_1500004903881.jpg',
 'Naturalistic3_1500005108073.jpg',
 'Naturalistic3_1500005119948.jpg',
 'Naturalistic3_1500005261833.jpg',
 'Naturalistic3_1500005273845.jpg',
 'Naturalistic3_1500005309788.jpg',
 'Naturalistic3_1500005407713.jpg',
 'Naturalistic3_1500005610361.jpg',
 'Naturalistic3_1500007083712.jpg',
 'Naturalistic3_1500007408817.jpg',
 'Naturalistic3_1500013448595.jpg',
 'Naturalistic3_1500017191516.jpg',
 'Naturalistic3_1502187405839.jpg',
 'Naturalistic3_1502188358954.jpg',
 'Naturalistic3_1502190317928.jpg',
 'Naturalistic3_1502190340953.jpg',
 'Naturalistic3_1502230118094.jpg',
 'Naturalistic3_1502230222884.jpg',
 'Naturalistic3_1502232417768.jpg',
 'Naturalistic3_1502235932033.jpg',
 'Naturalistic3_1502236185331.jpg',
 'Naturalistic3_1502236231413.jpg',
 'Naturalistic3_1502241980310.jpg',
 'Naturalistic3_1502251719405.jpg',
 'Naturalistic3_1502251766835.jpg',
 'Naturalistic3_1502256274333.jpg',
 'Naturalistic3_1502268701190.jpg',
 'Naturalistic3_1502268991335.jpg',
 'Naturalistic3_1502271022147.jpg',
 'Naturalistic3_1502271484672.jpg',
 'Naturalistic3_1502354988965.jpg',
 'Naturalistic3_1502361928037.jpg',
 'Naturalistic3_1502362023912.jpg',
 'Naturalistic3_1502362995956.jpg',
 'Naturalistic3_1502363139500.jpg',
 'Naturalistic3_1502409220673.jpg',
 'Naturalistic3_1502410837264.jpg',
 'Naturalistic3_1502429684260.jpg',
 'Naturalistic3_1502445846324.jpg',
 'Naturalistic4_1496794561198.jpg',
 'Naturalistic4_1496841113093.jpg',
 'Naturalistic4_1496841190652.jpg',
 'Naturalistic4_1496841947868.jpg',
 'Naturalistic4_1496846915837.jpg',
 'Naturalistic4_1496876352305.jpg',
 'Naturalistic4_1496887105399.jpg',
 'Naturalistic4_1496887522747.jpg',
 'Naturalistic4_1496900532589.jpg',
 'Naturalistic4_1496927289214.jpg',
 'Naturalistic4_1496927956130.jpg',
 'Naturalistic4_1496991749332.jpg',
 'Naturalistic4_1496993189573.jpg',
 'Naturalistic4_1496993621491.jpg',
 'Naturalistic4_1497050901775.jpg',
 'Naturalistic4_1497052877222.jpg',
 'Naturalistic4_1497052958186.jpg',
 'Naturalistic4_1497055933736.jpg',
 'Naturalistic4_1497058571919.jpg',
 'Naturalistic4_1497059263873.jpg',
 'Naturalistic4_1497059367645.jpg',
 'Naturalistic4_1497063016070.jpg',
 'Naturalistic4_1497067337824.jpg',
 'Naturalistic4_1497068905580.jpg',
 'Naturalistic4_1497069078303.jpg',
 'Naturalistic4_1497084230648.jpg',
 'Naturalistic4_1497086358335.jpg',
 'Naturalistic4_1497087187134.jpg',
 'Naturalistic4_1497088540857.jpg',
 'Naturalistic4_1497088575521.jpg',
 'Naturalistic4_1497088644816.jpg',
 'Naturalistic4_1497088840986.jpg',
 'Naturalistic4_1497091270737.jpg',
 'Naturalistic4_1497092272602.jpg',
 'Naturalistic4_1497147121134.jpg',
 'Naturalistic4_1497150661077.jpg',
 'Naturalistic4_1497154528214.jpg',
 'Naturalistic4_1497177783031.jpg',
 'Naturalistic4_1497228341820.jpg',
 'Naturalistic4_1497228619281.jpg',
 'Naturalistic4_1497229860901.jpg',
 'Naturalistic4_1497230764492.jpg',
 'Naturalistic4_1497230799327.jpg',
 'Naturalistic4_1497230939777.jpg',
 'Naturalistic4_1497231179652.jpg',
 'Naturalistic4_1497231392795.jpg',
 'Naturalistic4_1497236196115.jpg',
 'Naturalistic4_1497240424845.jpg',
 'Naturalistic4_1497261786072.jpg',
 'Naturalistic4_1497271869665.jpg',
 'Naturalistic5_1496842823965.jpg',
 'Naturalistic5_1496843965154.jpg',
 'Naturalistic5_1496886310687.jpg',
 'Naturalistic5_1496886323105.jpg',
 'Naturalistic5_1496888877141.jpg',
 'Naturalistic5_1496889172278.jpg',
 'Naturalistic5_1496891427902.jpg',
 'Naturalistic5_1496892889546.jpg',
 'Naturalistic5_1496964943058.jpg',
 'Naturalistic5_1496993586953.jpg',
 'Naturalistic5_1497053177320.jpg',
 'Naturalistic5_1497053805408.jpg',
 'Naturalistic5_1497054464556.jpg',
 'Naturalistic5_1497064428652.jpg',
 'Naturalistic5_1497067303192.jpg',
 'Naturalistic5_1497085566291.jpg',
 'Naturalistic5_1497087302496.jpg',
 'Naturalistic5_1497087626742.jpg',
 'Naturalistic5_1497087836407.jpg',
 'Naturalistic5_1497093355447.jpg',
 'Naturalistic5_1497094018526.jpg',
 'Naturalistic5_1497145863553.jpg',
 'Naturalistic5_1497151664330.jpg',
 'Naturalistic5_1497152936700.jpg',
 'Naturalistic5_1497152948259.jpg',
 'Naturalistic5_1497153559468.jpg',
 'Naturalistic5_1497154413617.jpg',
 'Naturalistic5_1497154551162.jpg',
 'Naturalistic5_1497175475241.jpg',
 'Naturalistic5_1497176045702.jpg',
 'Naturalistic5_1497177437006.jpg',
 'Naturalistic5_1497178154670.jpg',
 'Naturalistic5_1497180943830.jpg',
 'Naturalistic5_1497228259806.jpg',
 'Naturalistic5_1497228631133.jpg',
 'Naturalistic5_1497231095131.jpg',
 'Naturalistic5_1497246260659.jpg',
 'Naturalistic5_1497309560563.jpg',
 'Naturalistic5_1497309678309.jpg',
 'Naturalistic5_1497309698858.jpg',
 'Naturalistic5_1497313632552.jpg',
 'Naturalistic5_1497314231612.jpg',
 'Naturalistic5_1497314488174.jpg',
 'Naturalistic5_1497336579069.jpg',
 'Naturalistic5_1497337198110.jpg',
 'Naturalistic5_1497338100072.jpg',
 'Naturalistic5_1497338732575.jpg',
 'Naturalistic5_1497339284489.jpg',
 'Naturalistic5_1497339386840.jpg',
 'Naturalistic5_1497342036443.jpg',
 'Naturalistic5_1497344659228.jpg',
 'Naturalistic5_1497347177993.jpg',
 'Naturalistic5_1497347327114.jpg',
 'Naturalistic5_1497354978406.jpg',
 'Naturalistic5_1497355019930.jpg',
 'Naturalistic5_1497355795063.jpg',
 'Naturalistic5_1497356890503.jpg',
 'Naturalistic5_1497357282251.jpg',
 'Naturalistic5_1497394749158.jpg',
 'Naturalistic5_1497399395111.jpg',
 'Naturalistic5_1497401351445.jpg',
 'Naturalistic5_1497403767604.jpg',
 'Naturalistic5_1497403780412.jpg',
 'Naturalistic5_1497404628554.jpg',
 'Naturalistic5_1497406014648.jpg',
 'Naturalistic5_1497423050091.jpg',
 'Naturalistic5_1497425553152.jpg',
 'Naturalistic5_1497434778278.jpg',
 'Naturalistic5_1497487131165.jpg',
 'Naturalistic5_1497487152615.jpg',
 'Naturalistic5_1497487218354.jpg',
 'Naturalistic5_1497490565277.jpg',
 'Naturalistic5_1497490803552.jpg',
 'Naturalistic5_1497490838403.jpg',
 'Naturalistic5_1497491446773.jpg',
 'Naturalistic5_1497491845806.jpg',
 'Naturalistic5_1497491949577.jpg',
 'Naturalistic5_1497492394084.jpg',
 'Naturalistic5_1497492647756.jpg',
 'Naturalistic5_1497492693745.jpg',
 'Naturalistic6_1496728971354.jpg',
 'Naturalistic6_1496793883119.jpg',
 'Naturalistic6_1496795772097.jpg',
 'Naturalistic6_1496841007641.jpg',
 'Naturalistic6_1496841970893.jpg',
 'Naturalistic6_1496844887740.jpg',
 'Naturalistic6_1496876707265.jpg',
 'Naturalistic6_1496886126529.jpg',
 'Naturalistic6_1496887051642.jpg',
 'Naturalistic6_1496887392237.jpg',
 'Naturalistic6_1496888007783.jpg',
 'Naturalistic6_1496888020341.jpg',
 'Naturalistic6_1496888435770.jpg',
 'Naturalistic6_1496890894443.jpg',
 'Naturalistic6_1496911192932.jpg',
 'Naturalistic6_1496916190937.jpg',
 'Naturalistic6_1496991969246.jpg',
 'Naturalistic6_1497015053639.jpg',
 'Naturalistic6_1497050536161.jpg',
 'Naturalistic6_1497054186267.jpg',
 'Naturalistic6_1497055746676.jpg',
 'Naturalistic6_1497055969194.jpg',
 'Naturalistic6_1497058917990.jpg',
 'Naturalistic6_1497059310049.jpg',
 'Naturalistic6_1497061288024.jpg',
 'Naturalistic6_1497062704787.jpg',
 'Naturalistic6_1497063655624.jpg',
 'Naturalistic6_1497065367415.jpg',
 'Naturalistic6_1497065913650.jpg',
 'Naturalistic6_1497067786761.jpg',
 'Naturalistic6_1497067971559.jpg',
 'Naturalistic6_1497068536327.jpg',
 'Naturalistic6_1497084461606.jpg',
 'Naturalistic6_1497084634657.jpg',
 'Naturalistic6_1497085994714.jpg',
 'Naturalistic6_1497087418420.jpg',
 'Naturalistic6_1497088794841.jpg',
 'Naturalistic6_1497089626791.jpg',
 'Naturalistic6_1497090733597.jpg',
 'Naturalistic6_1497091846706.jpg',
 'Naturalistic6_1497092238048.jpg',
 'Naturalistic6_1497092825030.jpg',
 'Naturalistic6_1497092859584.jpg',
 'Naturalistic6_1497094456762.jpg',
 'Naturalistic6_1497096117494.jpg',
 'Naturalistic6_1497145541195.jpg',
 'Naturalistic6_1497145944455.jpg',
 'Naturalistic6_1497150398575.jpg',
 'Naturalistic6_1497150615025.jpg',
 'Naturalistic6_1497154251922.jpg',
 'Naturalistic6_1497217826958.jpg',
 'Naturalistic6_1497226920304.jpg',
 'Naturalistic6_1497231046356.jpg',
 'Naturalistic6_1497246283560.jpg',
 'Naturalistic6_1497246329690.jpg',
 'Naturalistic6_1497256574072.jpg',
 'Naturalistic6_1497258979377.jpg',
 'Naturalistic6_1497261046958.jpg',
 'Naturalistic6_1497271734166.jpg',
 'Naturalistic6_1497309627083.jpg',
 'Naturalistic6_1497310338247.jpg',
 'Naturalistic6_1497322206992.jpg',
 'Naturalistic6_1497324579381.jpg',
 'Naturalistic6_1497334319155.jpg',
 'Naturalistic6_1497335041808.jpg',
 'Naturalistic6_1497336873551.jpg',
 'Naturalistic6_1497337791816.jpg',
 'Naturalistic6_1497338359859.jpg',
 'Naturalistic6_1497342670724.jpg',
 'Naturalistic6_1497343205680.jpg',
 'Naturalistic6_1497343806469.jpg',
 'Naturalistic6_1497344648480.jpg',
 'Naturalistic6_1497347264636.jpg',
 'Naturalistic6_1497353459863.jpg',
 'Naturalistic6_1497355902253.jpg',
 'Naturalistic6_1497358089584.jpg',
 'Naturalistic6_1497399841100.jpg',
 'Naturalistic6_1497403409942.jpg',
 'Naturalistic6_1497403793672.jpg',
 'Naturalistic6_1497403819599.jpg',
 'TSEA1_1496789583706.jpg',
 'TSEA1_1496847099948.jpg',
 'TSEA1_1496876160962.jpg',
 'TSEA1_1496886272966.jpg',
 'TSEA1_1496886373665.jpg',
 'TSEA1_1496887158018.jpg',
 'TSEA1_1496889070878.jpg',
 'TSEA1_1496916450600.jpg',
 'TSEA1_1496968716174.jpg',
 'TSEA1_1496991946111.jpg',
 'TSEA1_1496993914585.jpg',
 'TSEA1_1496999962496.jpg',
 'TSEA1_1497050685028.jpg',
 'TSEA1_1497055367252.jpg',
 'TSEA1_1497058548799.jpg',
 'TSEA1_1497060414204.jpg',
 'TSEA1_1497084300052.jpg']

In [7]:
from fusefilter import fuse_heatmap, heatmap_filter

iou_thre = 0.5
ratio_mi = 0.5 # ratio_cd = 1-ratio_mi
kernel_pram = 80
thresh_pram = 80 # percentile, from small to big

ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"
DATA_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched')

save_path = os.path.join(ROOT_DIR, 'results_pad_test_final_eval')

if not os.path.exists(save_path):
    os.makedirs(save_path)

dirs = ['Test']

patch_types = [
    'Naturalistic1', 'Naturalistic2', 'Naturalistic3',
    'Naturalistic4', 'Naturalistic5', 'Naturalistic6',
    'TSEA1', 'TSEA2'
]

def get_mask(image, mask_generator):
    
    masks = mask_generator.generate(image.astype(np.uint8))
    return masks

if __name__ == "__main__":
    device = "cuda:0"
    # sam = sam_model_registry["vit_b"](checkpoint="models/sam_vit_b_01ec64.pth")
    torch.cuda.empty_cache()
    sam = sam_model_registry["vit_b"](checkpoint="../sam_vit_b_01ec64.pth")
    sam.to(device=device)
    mask_generator = SamAutomaticMaskGenerator(sam)

    print(save_path)
    folder = os.path.exists(save_path)
    
 #    processed_files = ['Naturalistic1_1499917703635.jpg',
 # 'Naturalistic1_1499918339461.jpg',
 # 'Naturalistic1_1499923500376.jpg',
 # 'Naturalistic1_1499925371739.jpg',
 # 'Naturalistic1_1499931818829.jpg']
    
    if not folder:
        os.makedirs(save_path)

    with torch.no_grad():
        for d_type in dirs:
            for patch in patch_types:
                input_path = os.path.join(DATA_DIR, d_type, patch, 'images')
                data_dir = input_path
                data_files = os.listdir(data_dir)
                start = time.time()
                # processed_files = [] # <-------------
                i = len(processed_files)
                for data_file in data_files:
                    print(data_file)
                    if data_file in processed_files:
                        continue
                    name = data_file.split(".")[0]
                    impath = os.path.join(data_dir, data_file)
                    
                    ori_img = Image.open(impath).convert('RGB')
                    ori_width, ori_height = ori_img.size
                    print("ori_height , ori_width", ori_height, ori_width)
        
                    mi_img, cd_img, fuse_img = fuse_heatmap(impath, ori_height, ori_width)
        
                    threshold = np.percentile(fuse_img, thresh_pram)
                    h_t, h_t_o, h_t_o_c, h_t_o_c_o = heatmap_filter(fuse_img, threshold, ori_height, ori_width)
        
                    gray = np.where(h_t_o_c_o >0,1,0)
        
                    rgb_color = cv2.imread(impath)
        
                    image = cv2.cvtColor(rgb_color, cv2.COLOR_BGR2RGB)
                    
                    #just for Dpatch
                    #image = cv2.resize(image,(416,416))
        
                    h = image.shape[0]
                    w = image.shape[1]
                    mask = get_mask(image, mask_generator)
        
                    result_mask = np.zeros((h,w))
                    for k in range(len(mask)):
        
                        mask_k = mask[k].get('segmentation')
                        n = mask_k&gray
                        u = mask_k #|gray
                        iou = np.sum(n)/(np.sum(u))
                        print("iou",iou)
        
                        n_1 = mask_k&result_mask.astype(np.uint8)
                        u_1 = mask_k
                        iou1 =  np.sum(n_1)/(np.sum(u_1))
                        print("iou1",iou1)
        
                        if(iou>iou_thre and iou1<0.1):
                            mask_k_save = np.expand_dims(mask_k,axis=2)
                            mask_k_save = np.tile(mask_k_save,3)
                            rgb_color = rgb_color*(~mask_k_save) 
                            result_mask = result_mask.astype(np.uint8) | mask_k
                            '''mask_k_save = np.expand_dims(mask_k,axis=2)
                            mask_k_save = np.tile(mask_k_save,3)
                            mask_gray = np.expand_dims(mask_k*128,axis=2)
                            mask_gray = np.tile(mask_gray,3)
                            rgb_color = rgb_color*(~mask_k_save) + mask_gray
                            result_mask = result_mask.astype(np.uint8) | mask_k'''
                            '''result_mask = result_mask.astype(np.uint8) | mask_k
                            rgb_color = cv2.inpaint(rgb_color, mask_k.astype(np.uint8), 3, cv2.INPAINT_NS)'''
        
                    
                    save_heatmap(h_t_o_c_o, os.path.join(save_path, name + "_h_t_o_c_o.png"))
                    save_heatmap(result_mask, os.path.join(save_path, name + "_pad_mask.png"))
                    # cv2.imwrite(os.path.join(save_path, name+".png"),rgb_color)
                    i+=1
                    elapsed = time.time() - start
                    print(f"{i} images processed. {elapsed:.2f} seconds elasped.")
                    processed_files.append(data_file)

C:\Adrianov\Projects\Project-Satanael\.venv311\Lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


C:\Adrianov\Projects\Project-Satanael\results_pad_test_final_eval
Naturalistic1_1499917703635.jpg
Naturalistic1_1499918339461.jpg
Naturalistic1_1499923500376.jpg
Naturalistic1_1499925371739.jpg
Naturalistic1_1499931818829.jpg
Naturalistic1_1499932367119.jpg
Naturalistic1_1499933971246.jpg
Naturalistic1_1499934217508.jpg
Naturalistic1_1499934495500.jpg
Naturalistic1_1499939610334.jpg
Naturalistic1_1499940414097.jpg
Naturalistic1_1499940503860.jpg
Naturalistic1_1499940647224.jpg
Naturalistic1_1499991199874.jpg
Naturalistic1_1499995387796.jpg
Naturalistic1_1499997243748.jpg
Naturalistic1_1499997506250.jpg
Naturalistic1_1499997891867.jpg
Naturalistic1_1499998395606.jpg
Naturalistic1_1499998680556.jpg
Naturalistic1_1499998720024.jpg
Naturalistic1_1499999051151.jpg
Naturalistic1_1499999078373.jpg
Naturalistic1_1500000581873.jpg
Naturalistic1_1500001935862.jpg
Naturalistic1_1500002615319.jpg
Naturalistic1_1500004891513.jpg
Naturalistic1_1500004915796.jpg
Naturalistic1_1500005214593.jpg
Natura

In [16]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())


12.1
True


In [18]:
!nvidia-smi


Tue Sep 23 14:24:34 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.90                 Driver Version: 565.90         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   35C    P8              1W /   75W |    5891MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
import gc, torch

# del sam        # or your model variable
# del mask_generator
torch.cuda.empty_cache()
gc.collect()


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [27]:
import os
import re
results_dir = r"C:\Adrianov\Projects\Project-Satanael\pad_res"

# Get all file names in the directory
files = os.listdir(results_dir)

unique_bases = set()

for f in files:
    match = re.match(r"(.+?_pad_mask\.png)$", f)
    if match:
        unique_bases.add(match.group(1))
        
processed_files = list(unique_bases)
processed_files = [f.replace("_pad_mask.png", ".jpg") for f in processed_files]
print(processed_files)
print(len(processed_files))

['1499563559079.jpg', '1499569799136.jpg', '1499563792315.jpg', '1499563803890.jpg', '1499565299106.jpg', '1499568775385.jpg', '1499564901648.jpg', '1499565240325.jpg', '1499565215708.jpg', '1499564959353.jpg', '1499563861719.jpg']
11


In [1]:
processed_files

NameError: name 'processed_files' is not defined

## End-to-End Inference for proposed method

In [11]:
import sys
import os
import importlib.util
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import warnings
from skimage import io, transform
from skimage.util import img_as_ubyte

compressdiff_path = os.path.abspath("../defenselib/spatial_heterogeneity.py")

spec = importlib.util.spec_from_file_location("spatial_heterogeneity", compressdiff_path)
compressdiff = importlib.util.module_from_spec(spec)
sys.modules["compressdiff"] = compressdiff
spec.loader.exec_module(compressdiff)

In [12]:
def save_heatmap(img, path, cmap='grey'):
    plt.imsave(path, img, cmap=cmap)

In [13]:
kernel_pram = 60

filenames_combi = []
ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"
TJUDHD_TRAIN_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger_test', 'images')
i = len(filenames_combi)


savefig_path = os.path.join(ROOT_DIR, 'results_cd_grey_test')

if not os.path.exists(savefig_path):
    os.makedirs(savefig_path)

start = time.time()
for root, _, files in os.walk(TJUDHD_TRAIN_DIR):
    for file in files:
        if (file.lower() in filenames_combi):
            continue
        if file.lower().endswith('.jpg'):
            print(file)
            
            impath = os.path.join(TJUDHD_TRAIN_DIR, file)
            
            OutputMap, OutputX = compressdiff.img_heatmap_cd(impath)
            average_OutputMap = np.mean(OutputMap, axis=0)
            OutputMap_max = np.max(average_OutputMap)
            OutputMap_min = np.min(average_OutputMap)
            out_height = len(average_OutputMap)
            out_width = len(average_OutputMap[0])
            average_OutputMap = [int((average_OutputMap[i][j]-OutputMap_min)*255/(OutputMap_max-OutputMap_min)) for i in range(out_height) for j in range(out_width)]
            flatNumpyArray = np.array(average_OutputMap,dtype=np.uint8)
            
            # Convert the array to make a grayscale image
            grayImage = flatNumpyArray.reshape(out_height, out_width)
            img = cv2.imread(impath)
            ori_height, ori_width, _ = img.shape
            grayImage = cv2.resize(grayImage, (ori_width, ori_height)) 

            # Morphological processing
            base_kernel_size = int(min(ori_height, ori_width)/kernel_pram)
            kernel=np.ones((base_kernel_size*2,base_kernel_size*2),np.uint8)
            opened = cv2.morphologyEx(grayImage, cv2.MORPH_OPEN,kernel, iterations=1)
            kernel=np.ones((base_kernel_size,base_kernel_size),np.uint8)
            closed=cv2.morphologyEx(opened,cv2.MORPH_CLOSE,kernel, iterations=2)
            kernel=np.ones((base_kernel_size*3,base_kernel_size*3),np.uint8)
            opened2=cv2.morphologyEx(closed,cv2.MORPH_OPEN,kernel, iterations=2)
            
            _, thresh_map_adversarial = cv2.threshold(opened2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            
            save_heatmap(grayImage, os.path.join(savefig_path, file + "_cd.png"))
            save_heatmap(opened, os.path.join(savefig_path, file + "_cd_o.png"))
            save_heatmap(closed, os.path.join(savefig_path, file + "_cd_o_c.png"))
            save_heatmap(opened2, os.path.join(savefig_path, file + "_cd_o_c_o.png"))
            save_heatmap(thresh_map_adversarial, os.path.join(savefig_path, file + "_cd_thresh.png"))
            i += 1
            elapsed = time.time() - start
            print(f"File {file} saved. {i} heatmap(s) processed. {elapsed:.2f} seconds elasped. Avg inference time: {(elapsed)/i:.2f}")
            filenames_combi.append(file)

seg_time = elapsed

1499563559079.jpg
height , width 1200 1624
File 1499563559079.jpg saved. 1 heatmap(s) processed. 37.28 seconds elasped. Avg inference time: 37.28
1499563792315.jpg
height , width 1200 1624
File 1499563792315.jpg saved. 2 heatmap(s) processed. 78.10 seconds elasped. Avg inference time: 39.05
1499563803890.jpg
height , width 1200 1624
File 1499563803890.jpg saved. 3 heatmap(s) processed. 117.06 seconds elasped. Avg inference time: 39.02
1499563861719.jpg
height , width 1200 1624
File 1499563861719.jpg saved. 4 heatmap(s) processed. 160.54 seconds elasped. Avg inference time: 40.13
1499564901648.jpg
height , width 1200 1624
File 1499564901648.jpg saved. 5 heatmap(s) processed. 202.28 seconds elasped. Avg inference time: 40.46
1499564959353.jpg
height , width 1200 1624
File 1499564959353.jpg saved. 6 heatmap(s) processed. 239.84 seconds elasped. Avg inference time: 39.97
1499565215708.jpg
height , width 1200 1624
File 1499565215708.jpg saved. 7 heatmap(s) processed. 282.61 seconds elasped.

In [14]:
seg_time

8575.785578727722